# 10 — Embedding signals into states

To *do* quantum optimal transport on a signal, you first face a modelling choice that decides everything downstream: **how does a signal become a quantum state?** This notebook lays out the embedding menu, writes each state by hand, builds the one object the capstone feeds to every quantum measure (the time-averaged joint state of two signals), and names exactly what each step keeps and what it throws away.

**Prerequisites:** `04/09_the_qot_zoo` (the three quantum optimal-transport objects, which take states and return numbers), `01/13_from_states_to_density_matrix` and `01/14_pure_vs_mixed` (a density matrix as an average of projectors), `01/17_tensor_and_partial_trace` (the tensor product and the partial trace), `01/19_entanglement` (separability and the partial-transpose test, used in the Challenge).

**What you'll be able to do**
- Explain why the embedding is a **modelling decision**, not a technicality.
- Build the three per-sample embeddings of `qot_course.quantum_ot` on a toy phase-and-amplitude signal — the **phase qubit**, the **amplitude–phase** qubit and the **multi-frequency** qutrit — write one sample of each by hand, and confirm each is a valid quantum state.
- Derive the **covariance density** of a record by hand, read its eigenvalue gap as the modulus of $\langle e^{2i\theta}\rangle$, and show that it cannot tell a phase $\theta$ from $\theta + \pi$.
- Define the **time-averaged joint state** $\rho_{AB}$ of two embedded signals as a mixture of product states, write its entries by hand, and build it with `joint_density_matrix` and `joint_density_from_states`.
- Say what the joint state keeps (how the two signals sit together at the same instant) and what it discards (the order of the samples in time).

Today you build the bridge from that machinery to real signals, and you will see that the bridge itself carries a scientific decision, which by the end of today you will make deliberately. Choose the embedding well and the geometry can see the structure that matters, together with the classical statistic that reads the same structure; choose it carelessly and it sees no more than the phase-locking value (PLV).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS, CMAP_PLAN
from qot_course.quantum.composite import partial_trace
from qot_course.quantum_ot.capstone import phase_qubit_state, joint_density_matrix
from qot_course.quantum_ot.embeddings import (
    amplitude_phase_state,
    covariance_density,
    multifreq_state,
    joint_density_from_states,
)

# One generator, seeded once: it draws the phase spreads of section 3 and the second
# signals of section 4, so every printed number below is reproducible.
rng = np.random.default_rng(42)
viz.use_course_style()

## 1. The embedding is a modelling decision

In `04/09` you put the three quantum optimal-transport objects on one bench and pinned down exactly how they relate. Those objects are now *fixed*: the coupling SDP, the entropic plan, the Bures bridge — each takes a quantum state (or a pair of them) and returns a number.

But a recorded signal — an EEG trace, an oscillator's phase — is **not** a quantum state. So before any QOT object can read it, we must decide how the signal *becomes* a state. That decision is the entire content of this notebook.

An embedding is a lossy summary. It cannot keep everything about a signal in a small density matrix, so it *chooses* what to keep. Two different embeddings of the same signal are two different states, and the same transport measure will generally return two different numbers for them — not because the measure changed, but because you asked it about different objects. This is not a flaw to be engineered away; it is the place where your scientific judgement enters.

The loss happens in two places. The first is the map from a signal to a state: section 2 writes three per-sample maps, and section 3 a fourth that summarises a whole record. The second is the time average that joins two embedded signals into one state: section 4 builds it and shows what it forgets.

## 2. Three ways to turn one sample into a state

Here is the menu the course offers, built on one toy signal so the contrasts are concrete. The signal is a phase $\theta(t)$ that drifts in time, carrying an amplitude envelope $a(t) \in [0, 1]$ alongside it — the two quantities a band-limited neural signal hands you after a Hilbert transform (instantaneous phase and instantaneous amplitude). Three embeddings turn each sample into its own pure state:

- **Phase qubit** (`phase_qubit_state`, the naive baseline): $\;(\lvert 0\rangle + e^{i\theta}\lvert 1\rangle)/\sqrt{2}$. The two basis weights are *fixed* at $1/\sqrt{2}$; only the phase enters, through the relative phase of the $\lvert 1\rangle$ component. On the Bloch sphere of `01/03` this is a point on the equator at azimuth $\theta$. It keeps the phase and **nothing else** — in particular it is blind to amplitude.
- **Amplitude–phase qubit** (`amplitude_phase_state`): $\;\cos\alpha\,\lvert 0\rangle + \sin\alpha\,e^{i\theta}\lvert 1\rangle$, where the polar angle $\alpha = \tfrac{\pi}{2}\,a$ carries the amplitude envelope. Now the $\lvert 1\rangle$ weight $\sin^2\alpha$ *tracks band power*, so a flat-power and a bursty signal with identical phases become **different** states.
- **Multi-frequency qutrit** (`multifreq_state`): $\;\tfrac{1}{\sqrt{3}}\bigl(\lvert 0\rangle + e^{i\theta}\lvert 1\rangle + e^{2i\theta}\lvert 2\rangle\bigr)$. It stacks the phase *and its second harmonic*: the slot $\lvert 2\rangle$ turns twice as fast as the slot $\lvert 1\rangle$.

**By hand, before the code.** Each state must have unit norm. For the phase qubit, $\lvert 1/\sqrt2\rvert^2 + \lvert e^{i\theta}/\sqrt2\rvert^2 = \tfrac12 + \tfrac12 = 1$, because a phase factor has modulus one. For the amplitude–phase qubit, $\cos^2\alpha + \sin^2\alpha = 1$. For the qutrit, three terms of $\tfrac13$ each. Now take one sample: at $t = 0.125$ the toy has $a = 1$, so $\alpha = \pi/2$ and the amplitude–phase state is $0\cdot\lvert 0\rangle + e^{i\theta}\lvert 1\rangle$. That is $\lvert 1\rangle$ times a global phase, and a global phase is invisible in any measurement (`01/02`): **at full amplitude this embedding has lost the phase.** The phase can only be read through the coherence $\lvert\langle 0\rvert\rho\lvert 1\rangle\rvert = \cos\alpha\sin\alpha = \tfrac12\sin(\pi a)$ of the per-sample density matrix $\rho = \lvert\psi\rangle\langle\psi\rvert$, which is largest at $a = \tfrac12$ and vanishes at $a = 0$ and $a = 1$. The phase qubit keeps it at $\tfrac12$ for every sample: letting amplitude in has a price.

In [ ]:
# A toy signal: a drifting phase with an amplitude envelope in [0, 1].
n_samples = 600
t = np.linspace(0.0, 1.0, n_samples, endpoint=False)
carrier_hz = 8.0
theta = 2 * np.pi * carrier_hz * t + 0.4 * np.sin(2 * np.pi * 1.5 * t)  # phase drifts
amp = 0.5 + 0.5 * np.sin(2 * np.pi * 2.0 * t)                          # envelope in [0, 1]

# Build each per-sample embedding on the same signal.
psi_naive = phase_qubit_state(theta)                       # (n, 2) pure qubits
psi_ampphase = amplitude_phase_state(theta, amp)           # (n, 2) pure qubits
psi_multifreq = multifreq_state(theta, harmonics=(1, 2))   # (n, 3) pure qutrits


def per_sample_norms(psi):
    return np.sum(np.abs(psi) ** 2, axis=1)


for name, psi in [("phase qubit", psi_naive), ("amplitude-phase", psi_ampphase),
                  ("multi-frequency", psi_multifreq)]:
    print(f"{name:<17} shape {psi.shape}   all rows unit-norm: "
          f"{bool(np.allclose(per_sample_norms(psi), 1.0))}")

# One sample by hand: t = 0.125, where the envelope is at its top (a = 1).
k = 75
print()
print(f"sample at t = {t[k]:.3f}: a = {amp[k]:.4f}, theta mod 2pi = {np.mod(theta[k], 2 * np.pi):.4f}")
print("  phase qubit      ", np.round(psi_naive[k], 4))
print("  amplitude-phase  ", np.round(psi_ampphase[k], 4))
print("  multi-frequency  ", np.round(psi_multifreq[k], 4))

# The defining contrast between the two qubits: the |1>-weight and the phase-carrying coherence.
weight_one_naive = np.abs(psi_naive[:, 1]) ** 2
weight_one_ampphase = np.abs(psi_ampphase[:, 1]) ** 2
coh_naive = np.abs(psi_naive[:, 0] * np.conj(psi_naive[:, 1]))          # |<0|rho|1>| per sample
coh_ampphase = np.abs(psi_ampphase[:, 0] * np.conj(psi_ampphase[:, 1]))
print()
print(f"|1>-weight:  phase qubit {weight_one_naive.min():.4f} to {weight_one_naive.max():.4f};  "
      f"amplitude-phase {weight_one_ampphase.min():.4f} to {weight_one_ampphase.max():.4f}")
print("amplitude-phase |1>-weight equals sin^2(pi a / 2):",
      bool(np.allclose(weight_one_ampphase, np.sin(np.pi * amp / 2) ** 2)))
print(f"coherence:   phase qubit {coh_naive.min():.4f} to {coh_naive.max():.4f};  "
      f"amplitude-phase {coh_ampphase.min():.4f} to {coh_ampphase.max():.4f}")
print("amplitude-phase coherence equals sin(pi a) / 2:",
      bool(np.allclose(coh_ampphase, 0.5 * np.sin(np.pi * amp))))

**Read the output.** Every row of every embedding is unit-norm, so each sample is a legitimate pure state, in dimension $2$ for the qubits and $3$ for the qutrit.

The sample at $t = 0.125$ is the hand computation of the section. The phase qubit prints two entries of modulus $0.7071 = 1/\sqrt2$, the second one carrying the phase. The amplitude–phase qubit prints $0$ on $\lvert 0\rangle$ and a phase factor on $\lvert 1\rangle$: at the top of the envelope it has become $\lvert 1\rangle$ up to a global phase. The qutrit prints three entries of modulus $0.5774 = 1/\sqrt3$.

The last four lines hold the heart of the menu. The phase qubit's $\lvert 1\rangle$-weight is pinned at $0.5$ and its coherence at $0.5$, for every sample. The amplitude–phase qubit's $\lvert 1\rangle$-weight sweeps the full range from $0$ to $1$ and matches $\sin^2(\pi a/2)$ everywhere, while its coherence sweeps from $0$ to $0.5$ and matches $\tfrac12\sin(\pi a)$. Same signal, different states, because each embedding made a different choice about what to keep.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), gridspec_kw={"wspace": 0.3})

# (a) The toy phase signal itself (wrapped), for context.
ax = axes[0]
ax.plot(t, np.mod(theta, 2 * np.pi), color=COLORS["flow"], lw=1.8)
ax.set_xlabel("time  $t$  (s)")
ax.set_ylabel(r"phase  $\theta(t)\ \mathrm{mod}\ 2\pi$")
ax.set_title("The toy signal: a drifting phase", pad=10)

# (b) The two qubits: |1>-weight and phase-carrying coherence over time.
ax = axes[1]
ax.plot(t, weight_one_naive, color=COLORS["quantum"], lw=2.2, label="phase qubit")
ax.plot(t, weight_one_ampphase, color=COLORS["highlight"], lw=2.2, label=r"amp.-phase $|1\rangle$-weight")
ax.plot(t, coh_ampphase, color=COLORS["source"], lw=2.0, ls=":", label=r"amp.-phase coherence")
ax.plot(t, amp, color=COLORS["muted"], lw=1.4, ls="--", label="envelope $a(t)$")
ax.set_xlabel("time  $t$  (s)")
ax.set_ylabel("value")
ax.set_ylim(-0.05, 1.45)
ax.set_title("What each qubit lets in", pad=10)
ax.legend(loc="upper center", fontsize=8, ncol=2)

# (c) The qutrit's two phase slots over the first quarter second: |2> turns twice as fast.
ax = axes[2]
window = t < 0.25
ax.plot(t[window], np.angle(psi_multifreq[window, 1]), color=COLORS["quantum"], lw=2.0,
        label=r"slot $|1\rangle$: phase $\theta$")
ax.plot(t[window], np.angle(psi_multifreq[window, 2]), color=COLORS["highlight"], lw=2.0,
        label=r"slot $|2\rangle$: phase $2\theta$")
ax.set_xlabel("time  $t$  (s)")
ax.set_ylabel(r"phase of the slot, in $(-\pi, \pi]$")
ax.set_ylim(-3.4, 4.6)
ax.set_title("Qutrit: one slot per harmonic", pad=10)
ax.legend(loc="upper center", fontsize=8, ncol=2)

plt.show()

**Read the figure.** *Left* is the toy phase, wrapped into $[0, 2\pi)$: eight turns in one second, the trace all three embeddings read.

*Middle* contrasts the two qubits. The sky-blue line is the phase qubit: its $\lvert 1\rangle$-weight and its coherence are both flat at $\tfrac12$, no matter what the envelope does. The rose line, the amplitude–phase $\lvert 1\rangle$-weight, rises and falls with the dashed envelope, touching $0$ when $a = 0$ and $1$ when $a = 1$. The dotted periwinkle line is the price: the coherence that carries the phase peaks where the envelope crosses $\tfrac12$ and drops to zero at both extremes of the envelope.

*Right* shows the qutrit's two phase slots over the first quarter second. The rose sawtooth (slot $\lvert 2\rangle$) wraps twice for every wrap of the sky-blue one (slot $\lvert 1\rangle$): the second slot carries $2\theta$. On one signal alone it adds nothing new, since $2\theta$ is a function of $\theta$. Its value appears only when two signals are joined, which is the subject of `04/11`.

## 3. The covariance density: one state for a whole record

The fourth item of the menu works differently. `covariance_density` does not return one state per sample; it returns a single $2\times 2$ **mixed** state for the whole record. You can derive it by hand. Send each sample to the *real* unit vector $\lvert v(\theta)\rangle = \cos\theta\,\lvert 0\rangle + \sin\theta\,\lvert 1\rangle$ and average its projector over the record. With the double-angle formulas $\cos^2\theta = \tfrac12(1 + \cos 2\theta)$, $\sin^2\theta = \tfrac12(1 - \cos 2\theta)$ and $\cos\theta\sin\theta = \tfrac12\sin 2\theta$,

$$ \lvert v\rangle\langle v\rvert = \frac12\begin{pmatrix} 1 + \cos 2\theta & \sin 2\theta \\ \sin 2\theta & 1 - \cos 2\theta \end{pmatrix}, \qquad \rho_{\mathrm{cov}} = \frac12\begin{pmatrix} 1 + C & S \\ S & 1 - C \end{pmatrix}, \quad C + iS = \langle e^{2i\theta}\rangle . $$

The trace is already $1$, so the library's normalisation changes nothing. The eigenvalues of this matrix are $\tfrac12\bigl(1 \pm \sqrt{C^2 + S^2}\bigr)$, so the **eigenvalue gap is exactly** $\lvert\langle e^{2i\theta}\rangle\rvert$: the modulus of the second circular moment of the phase (Mardia & Jupp 2000). The $m$-th **circular moment** of a phase is the average $\langle e^{im\theta}\rangle$ of the unit vectors $e^{im\theta}$; its modulus runs from $1$, when all samples point the same way, to $0$, when they cancel around the circle.

Two consequences follow.

- **What it keeps: the concentration of the phases.** For phases spread around a centre with a Gaussian spread of standard deviation $\sigma$, the average of $e^{ikX}$ over a normal variable of standard deviation $\sigma$ is $e^{ik\mu}e^{-k^2\sigma^2/2}$ (its characteristic function); at $k = 2$ its modulus is $e^{-2\sigma^2}$, and wrapping the phase onto the circle changes nothing because $e^{2i\theta}$ is $2\pi$-periodic. Tight phases give a gap near $1$ and a near-pure state; phases smeared around the circle give a gap near $0$ and the maximally mixed $\tfrac12 I$.
- **What it discards: the difference between $\theta$ and $\theta + \pi$.** Since $\lvert v(\theta + \pi)\rangle = -\lvert v(\theta)\rangle$, and an overall sign is a global phase, the two samples give the same projector. A record whose phases sit in one cluster and a record whose phases are split between that cluster and its antipode are **the same state** to this embedding, although one record's first circular moment is close to one and the other's close to zero.

Section 4 pairs the per-sample embeddings of section 2, not this one-record summary.

In [ ]:
# The hand formula against the library, on the toy signal of section 2.
m2_theta = np.mean(np.exp(2j * theta))
rho_by_hand = 0.5 * np.array([[1 + m2_theta.real, m2_theta.imag],
                              [m2_theta.imag, 1 - m2_theta.real]])
rho_cov = covariance_density(theta)
eig = np.sort(np.linalg.eigvalsh(rho_cov.real))[::-1]
print("covariance density of the toy signal (real part):")
print(np.round(rho_cov.real, 4))
print("agrees with the formula built from <e^(2i theta)>:", bool(np.allclose(rho_cov, rho_by_hand)))
print(f"eigenvalues {eig[0]:.4f} and {eig[1]:.4f};  gap {eig[0] - eig[1]:.4f};  "
      f"|<e^(2i theta)>| = {abs(m2_theta):.4f}")

# What it keeps: the gap against the Gaussian spread of the phases.
n_spread = 40_000
centre = np.pi / 4
spreads = np.array([0.3, 0.7, 1.2, 2.0])
gaps = []
print()
for sd in spreads:
    th = centre + sd * rng.standard_normal(n_spread)
    e = np.sort(np.linalg.eigvalsh(covariance_density(th).real))[::-1]
    gaps.append(e[0] - e[1])
    print(f"spread {sd:.1f} rad: eigenvalue gap {e[0] - e[1]:.4f}   closed form exp(-2 sigma^2) = {np.exp(-2 * sd**2):.4f}")
print(f"finite-sample floor, of order 1/sqrt(N) = {1 / np.sqrt(n_spread):.4f}")

# What it discards: theta and theta + pi give the same projector.
cluster = centre + 0.3 * rng.standard_normal(n_spread)
antipodal = cluster + np.pi * rng.integers(0, 2, n_spread)     # half the samples moved by pi
uniform = rng.uniform(0.0, 2 * np.pi, n_spread)
records = {"one cluster": cluster, "two antipodal clusters": antipodal, "uniform": uniform}
record_eigs = {}
print()
for name, th in records.items():
    e = np.sort(np.linalg.eigvalsh(covariance_density(th).real))[::-1]
    record_eigs[name] = e
    print(f"{name:<24} |<e^(i theta)>| = {abs(np.mean(np.exp(1j * th))):.4f}   eigenvalues {e[0]:.4f}, {e[1]:.4f}")
print("largest entry of rho_cov(one cluster) - rho_cov(two antipodal clusters):",
      float(np.abs(covariance_density(cluster) - covariance_density(antipodal)).max()))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
grid = np.linspace(0.0, 2.2, 200)
ax.plot(grid, np.exp(-2 * grid**2), color=COLORS["source"], lw=2.2, label=r"closed form $e^{-2\sigma^2}$")
ax.plot(spreads, gaps, "o", color=COLORS["highlight"], markersize=9, label=f"measured gap, N = {n_spread}")
ax.axhline(1 / np.sqrt(n_spread), color=COLORS["muted"], ls=":", lw=1.2, label=r"finite-sample floor, order $1/\sqrt{N}$")
ax.set_xlabel(r"phase spread $\sigma$ (radians)")
ax.set_ylabel("eigenvalue gap of the covariance density")
ax.set_title(r"The gap is $|\langle e^{2i\theta}\rangle|$", pad=10)
ax.legend(loc="upper right")

ax = axes[1]
names = list(records)
x = np.arange(len(names))
ax.bar(x - 0.19, [record_eigs[n][0] for n in names], width=0.38, color=COLORS["quantum"], label="larger eigenvalue")
ax.bar(x + 0.19, [record_eigs[n][1] for n in names], width=0.38, color=COLORS["target"], label="smaller eigenvalue")
ax.set_xticks(x)
ax.set_xticklabels(["one cluster", "two antipodal\nclusters", "uniform"])
ax.set_ylim(0, 1.15)
ax.set_ylabel("eigenvalue")
ax.set_title(r"Blind to $\theta \to \theta + \pi$", pad=10)
ax.legend(loc="upper right")
plt.show()

**Read the output and the figure.** The first block confirms the hand formula on the toy signal: the library's matrix and the one built from $\langle e^{2i\theta}\rangle$ agree, and the printed gap equals the printed modulus of the second moment. The toy's phases turn round the circle eight times, so that modulus is small and the state is close to $\tfrac12 I$.

*Left:* the measured gaps sit on the closed form $e^{-2\sigma^2}$ at the three smaller spreads: $0.8339$ against $0.8353$ at $\sigma = 0.3$, $0.3787$ against $0.3753$ at $\sigma = 0.7$, and $0.0550$ against $0.0561$ at $\sigma = 1.2$. At $\sigma = 2.0$ the closed form, $0.0003$, has dropped below the dotted line, the finite-sample floor of about $0.005$ at $N = 40{,}000$ (an average of $N$ unit vectors with random directions has a length of order $1/\sqrt N$, the step count of a random walk), and the measured gap, $0.0019$, is of the order of that floor instead: a finite record cannot resolve a second moment smaller than its own sampling noise.

*Right:* the one-cluster record and the two-antipodal-cluster record have the same two bars, a near-pure state with eigenvalues $0.9178$ and $0.0822$. The printed difference between the two matrices is exactly zero, while their first moments differ: $0.9561$ for one cluster and $0.0066$ for the split record. A uniform record gives two equal bars. The covariance density reads **axial** data, a direction without its sign: it throws the sign away by design.

## 4. The time-averaged joint state of two signals

Everything so far concerned one signal. A coupling measure needs two. Each per-sample embedding turns sample $t$ of two signals $A$ and $B$ into states $\lvert\psi_A(t)\rangle$ and $\lvert\psi_B(t)\rangle$, and the **joint state** is the time average of their product:

$$ \rho_{AB} \;=\; \frac{1}{N}\sum_{t=1}^{N} |\psi_A(t)\rangle\langle\psi_A(t)| \otimes |\psi_B(t)\rangle\langle\psi_B(t)|. $$

Every sample contributes a product state, but an average of products is in general not a product: $\rho_{AB}$ is a mixture of product states that carries the correlations between the two signals. `joint_density_matrix(theta_a, theta_b)` builds it for the phase qubit, and `joint_density_from_states` for any two arrays of embedded states. Trace one and positivity come for free, because $\rho_{AB}$ is an average of unit-trace projectors (`01/13`).

**Its entries, by hand, for the phase qubit.** The product state at sample $t$ has amplitude $\tfrac12 e^{\mathrm{i}(i\theta_A + j\theta_B)}$ on the basis vector $\lvert i\rangle_A\lvert j\rangle_B$, with $i, j \in \{0, 1\}$, which the library stores at flat index $2i + j$. Here, and wherever a basis index $i$ stands beside it, the upright $\mathrm{i}$ is the imaginary unit. Multiplying an amplitude by the conjugate of another and averaging,

$$ \rho_{AB}\bigl[(ij), (kl)\bigr] \;=\; \tfrac14\,\bigl\langle e^{\mathrm{i}[(i-k)\theta_A + (j-l)\theta_B]}\bigr\rangle . $$

So **every entry is one quarter of the time average of one phase factor** $e^{\mathrm{i}(m\theta_A + n\theta_B)}$, with $m = i - k$ and $n = j - l$ in $\{-1, 0, 1\}$. The diagonal ($m = n = 0$) is $\tfrac14$ everywhere. The partial trace over $B$ keeps the entries with $n = 0$ and gives the marginal $\rho_A = \tfrac12\begin{pmatrix} 1 & \langle e^{-i\theta_A}\rangle \\ \langle e^{i\theta_A}\rangle & 1\end{pmatrix}$, the time average of signal $A$'s own states.

**Two facts read off the formula.**

- *If the two signals are independent*, each average factorises, $\langle e^{i(m\theta_A + n\theta_B)}\rangle = \langle e^{im\theta_A}\rangle\langle e^{in\theta_B}\rangle$, and $\rho_{AB} = \rho_A \otimes \rho_B$. The coupling therefore lives in the gap between $\rho_{AB}$ and the product of its marginals, and that gap is what the quantum mutual information and the Bures coupling of the capstone measure (`04/13`).
- *Any reordering of the samples applied to both signals alike leaves $\rho_{AB}$ unchanged*, because a sum does not depend on the order of its terms. The joint state keeps how the two signals sit together **at the same instant**, sample by sample, and discards **when** each instant happened. Anything that lives in time order (a lag, a direction of influence, a dynamics) is out of its reach; `02/05` measured direction with transfer entropy, and that question stays a classical one here.

We check this on signal $A$, the drifting phase of section 2 over a longer record, with a *locked* partner that follows it at an offset of $0.7$ rad with jitter, and an *independent* partner on its own carrier.

In [ ]:
# A longer record, so that time averages settle: 10 s at the sampling rate of section 2.
n_joint = 20_000
tj = np.linspace(0.0, 10.0, n_joint, endpoint=False)
theta_a = 2 * np.pi * 8.0 * tj + 0.4 * np.sin(2 * np.pi * 1.5 * tj)
theta_locked = theta_a - 0.7 + 0.3 * rng.standard_normal(n_joint)                      # follows A, with jitter
theta_indep = 2 * np.pi * 11.3 * tj + np.cumsum(0.05 * rng.standard_normal(n_joint))  # its own carrier

rho_locked = joint_density_matrix(theta_a, theta_locked)
same = joint_density_from_states(phase_qubit_state(theta_a), phase_qubit_state(theta_locked))
print("joint_density_matrix and joint_density_from_states agree:", bool(np.allclose(rho_locked, same)))


def joint_by_hand(ta, tb):
    '''rho[(ij),(kl)] = (1/4) < exp(i[(i-k) theta_A + (j-l) theta_B]) >, at flat index 2i + j.'''
    rho = np.zeros((4, 4), dtype=complex)
    for i in range(2):
        for j in range(2):
            for k in range(2):
                for l in range(2):
                    rho[2 * i + j, 2 * k + l] = 0.25 * np.mean(np.exp(1j * ((i - k) * ta + (j - l) * tb)))
    return rho


print("largest gap between the entry formula and the library:",
      float(np.abs(joint_by_hand(theta_a, theta_locked) - rho_locked).max()))
print(f"trace {np.trace(rho_locked).real:.6f};  smallest eigenvalue {np.linalg.eigvalsh(rho_locked).min():.6f}")
print("marginal rho_A of the locked pair:")
print(np.round(partial_trace(rho_locked, keep=[0], dims=[2, 2]), 4))


def product_gap(rho):
    '''Frobenius distance between a two-qubit state and the product of its two marginals.'''
    rho_a = partial_trace(rho, keep=[0], dims=[2, 2])
    rho_b = partial_trace(rho, keep=[1], dims=[2, 2])
    return float(np.linalg.norm(rho - np.kron(rho_a, rho_b)))


order = rng.permutation(n_joint)
cases = {
    "locked pair": rho_locked,
    "locked, reordered alike": joint_density_matrix(theta_a[order], theta_locked[order]),
    "locked, B reordered": joint_density_matrix(theta_a, theta_locked[order]),
    "independent pair": joint_density_matrix(theta_a, theta_indep),
}
print()
for name, rho in cases.items():
    print(f"{name:<26} distance to rho_A (x) rho_B = {product_gap(rho):.4f}")
print("reordering both signals alike changes rho_AB by at most",
      float(np.abs(cases["locked, reordered alike"] - rho_locked).max()))
print(f"locked pair: |rho[1,2]| = {abs(rho_locked[1, 2]):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"wspace": 0.55})
labels = [r"$|00\rangle$", r"$|01\rangle$", r"$|10\rangle$", r"$|11\rangle$"]
for ax, key, title in [(axes[0], "locked pair", "Locked pair: $|\\rho_{AB}|$"),
                       (axes[1], "locked, B reordered", "B reordered alone: $|\\rho_{AB}|$")]:
    mag = np.abs(cases[key])
    ax.imshow(mag, cmap=CMAP_PLAN, vmin=0.0, vmax=0.25)
    for r in range(4):
        for c in range(4):
            ax.text(c, r, f"{mag[r, c]:.3f}", ha="center", va="center", fontsize=9, color=COLORS["text"])
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels)
    ax.set_yticks(range(4))
    ax.set_yticklabels(labels)
    ax.set_title(title, pad=10)
    ax.grid(False)

ax = axes[2]
names = list(cases)
values = [product_gap(cases[n]) for n in names]
bar_colors = [COLORS["highlight"], COLORS["highlight"], COLORS["quantum"], COLORS["quantum"]]
ax.barh(np.arange(len(names)), values, color=bar_colors)
ax.set_yticks(np.arange(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel(r"$\|\rho_{AB} - \rho_A \otimes \rho_B\|_F$")
ax.set_title("Distance from the product of the marginals", pad=10)
plt.show()

**Read the output and the figure.** The two library helpers agree, and the entry formula reproduces them to rounding error. The trace is one and no eigenvalue is negative, as for any average of projectors. The marginal $\rho_A$ is close to $\tfrac12 I$, because signal $A$ turns round the circle many times and $\langle e^{i\theta_A}\rangle$ nearly vanishes.

*Left:* the locked pair. The diagonal sits at $0.25$, as the formula says. Two off-diagonal entries stand out, the pair linking $\lvert 01\rangle$ and $\lvert 10\rangle$, at $0.239$; every other off-diagonal entry is close to zero. Those two entries are the only ones whose phase factor, $e^{\mp i(\theta_A - \theta_B)}$, does not turn with the carrier, so they are the only ones that survive the time average. `04/11` reads exactly what they hold.

*Middle:* the same pair after reordering signal $B$ alone. The diagonal is untouched, but the two bright entries are gone: breaking the sample-by-sample pairing has destroyed the correlation.

*Right:* the distance from $\rho_{AB}$ to $\rho_A\otimes\rho_B$ for the four cases. The locked pair and the locked pair reordered alike have the same bar, $0.338$, because the joint state is the same matrix to machine precision: the time order is not in it. Reordering $B$ alone gives $0.003$ and the independent pair $0.015$, both close to zero; what remains is the residue of a finite record.

## Your turn

**Warm-up.** Take $a = \tfrac12$ and $\theta = \pi/2$. Write the amplitude–phase state by hand, its density matrix and its coherence, and check with `amplitude_phase_state`. Then find the envelope values at which two samples with phases $\theta$ and $\theta + \pi$ give the *same* density matrix, and say what this embedding has lost there.

**Try it.** Predict, then check, the joint state of two extreme pairs. First take perfect locking, `theta_b = theta_a` with `theta_a = np.linspace(0, 2 * np.pi, 10_000, endpoint=False)` sweeping the circle: write $\rho_{AB}$ by hand from the entry formula of section 4, then compare with `joint_density_matrix(theta_a, theta_a)`. Which entries survive the time average, what is $\rho[1,2]$, and what are the four eigenvalues? Then destroy the locking with `theta_b = np.random.default_rng(0).permutation(theta_a)`. Which state does the joint state approach, and what does `product_gap` print for that pair?

**Go further.** Compute by hand the covariance density of a record whose phases are spread uniformly over the half circle $[0, \pi)$, then over the quarter circle $[0, \pi/2)$. Predict each eigenvalue gap from $\lvert\langle e^{2i\theta}\rangle\rvert$ and check with `covariance_density`. Which of the two records looks maximally mixed to the covariance density, although its phases cover only half the circle? Compute its first moment $\lvert\langle e^{i\theta}\rangle\rvert$ too, and say in one sentence what the embedding has thrown away.

**Challenge.** (a) Using the definition of separability of `01/19` (Werner 1989), say in one line why every $\rho_{AB}$ of section 4 is separable. (b) For a phase-qubit $\rho_{AB}$ whose only non-zero entries are the diagonal $\tfrac14$ and the pair $\rho[1,2] = \bar\rho[2,1] = c/4$, write the partial transpose on $B$ by hand: which position does the pair move to? Show that its eigenvalues are $\tfrac14, \tfrac14, \tfrac14(1 \pm \lvert c\rvert)$, so the smallest is $(1 - \mathrm{PLV})/4 \ge 0$, and check on `rho_locked` (reshape to `(2, 2, 2, 2)`, swap the indices of $B$), whose other off-diagonal entries are only the small residue of a finite record. For two qubits, as here, a partial transpose with no negative eigenvalue is also sufficient for separability; from $3\times3$ on it is only necessary (Horodecki, Horodecki & Horodecki 1996). (c) Set this beside the Bell state of `01/19`, whose partial transpose has $-\tfrac12$: which entries would $\rho_{AB}$ need, and how large, to go negative? Conclude what kind of correlation a quantum measure of $\rho_{AB}$ counts on these states: correlation without entanglement, though not necessarily of a purely classical kind, because the product states mixed here are not orthogonal.

## What you built

- You can explain why the embedding is a **modelling decision**: the QOT objects of `04/09` are fixed, so the state you build decides the number they return.
- You can build the three per-sample embeddings of `qot_course.quantum_ot` — the phase qubit, the amplitude–phase qubit and the multi-frequency qutrit — write one sample of each by hand, and confirm each is a valid state.
- You can say what each keeps: the phase only (the naive qubit); the phase and the band power, at the price of losing the phase at both extremes of the envelope (amplitude–phase); the phase at its first and second harmonic (multi-frequency).
- You can derive the covariance density of a record by hand, read its eigenvalue gap as $\lvert\langle e^{2i\theta}\rangle\rvert$, check it against $e^{-2\sigma^2}$, and show that it cannot tell $\theta$ from $\theta + \pi$.
- You can define the time-averaged joint state as a mixture of product states, write its entries as averages of phase factors, build it with `joint_density_matrix` and `joint_density_from_states`, and show that it equals the product of its marginals for independent signals and ignores the order of the samples.

Stand back: you can now choose an embedding *deliberately*, knowing what it lets the geometry see and what it hides, and you hold the one object every quantum measure of the capstone reads.

## What's next

Section 4 left two bright entries in the locked pair's joint state and a promise to read them. In `04/11_moments_the_joint_state_reads` you take the phase difference $\delta = \theta_A - \theta_B$ of two signals and find which of its circular moments each embedding's joint state holds: the phase qubit holds the first, which is what the phase-locking value (PLV) measures, and the multi-frequency qutrit holds the second as well. A small two-signal toy then shows what that extra entry can tell apart, and what the quantum mutual information makes of it.

## References

- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments; the second moment and axial data behind the covariance density.
- R. F. Werner, "Quantum states with Einstein-Podolsky-Rosen correlations admitting a hidden-variable model", *Physical Review A* **40**, 4277–4281 (1989). DOI:10.1103/PhysRevA.40.4277 — the definition of a separable state as a mixture of product states.
- A. Peres, "Separability criterion for density matrices", *Physical Review Letters* **77**, 1413–1415 (1996). DOI:10.1103/PhysRevLett.77.1413 — the partial-transpose test of the Challenge.
- M. Horodecki, P. Horodecki & R. Horodecki, "Separability of mixed states: necessary and sufficient conditions", *Physics Letters A* **223**, 1–8 (1996). DOI:10.1016/S0375-9601(96)00706-2 — the partial-transpose test is sufficient for two qubits, only necessary in larger dimensions.
- D. Trevisan, "Quantum optimal transport: an invitation", *Bollettino dell'Unione Matematica Italiana* **18**, 347–360 (2025). DOI:10.1007/s40574-024-00428-5

**Previous:** `notebooks/04_QuantumOptimalTransport/09_the_qot_zoo.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/11_moments_the_joint_state_reads.ipynb`